In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import polars as pl
from datetime import date

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
RAW_PATH = PROJECT_PATH / "data" / "raw"
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

In [3]:
transactions = pl.scan_parquet(PROCESSED_PATH / "transactions.parquet")
articles = pl.read_csv(RAW_PATH / "articles.csv")
customers = pl.read_csv(RAW_PATH / "customers.csv")

In [4]:
print("Transactions:", transactions.select(pl.len()).collect().item())
print("Customers:", customers.height)
print("Articles:", articles.height)

Transactions: 31788324
Customers: 1371980
Articles: 105542


In [5]:
customer_mapping = customers.select("customer_id").with_row_index("customer_idx", offset=1)
article_mapping = articles.select("article_id").with_row_index("article_idx", offset=1)

In [6]:
customer_mapping.head()

customer_idx,customer_id
u32,str
1,"""00000dbacae5abe5e23885899a1fa4…"
2,"""0000423b00ade91418cceaf3b26c6a…"
3,"""000058a12d5b43e67d225668fa1f8d…"
4,"""00005ca1c9ed5f5146b52ac8639a40…"
5,"""00006413d8573cd20ed7128e53b7b1…"


In [7]:
article_mapping.head()

article_idx,article_id
u32,i64
1,108775015
2,108775044
3,108775051
4,110065001
5,110065002


In [8]:
transactions_mapped = (
    transactions
    .join(customer_mapping.lazy(), on="customer_id")
    .join(article_mapping.lazy(), on="article_id")
    .select(
        "t_dat",
        "customer_idx",
        "article_idx",
        "price",
        "sales_channel_id"
    )
)

In [9]:
transactions_mapped.head().collect()

t_dat,customer_idx,article_idx,price,sales_channel_id
date,u32,u32,f32,i8
2018-09-20,3,40180,0.050831,2
2018-09-20,3,10521,0.030492,2
2018-09-20,8,6388,0.015237,2
2018-09-20,8,46305,0.016932,2
2018-09-20,8,46306,0.016932,2


In [10]:
print("До mapping:", transactions.select(pl.len()).collect().item())
print("После mapping:", transactions_mapped.select(pl.len()).collect().item())

До mapping: 31788324
После mapping: 31788324


In [11]:
customer_mapping.write_parquet(PROCESSED_PATH / "customer_mapping.parquet")
article_mapping.write_parquet(PROCESSED_PATH / "article_mapping.parquet")

In [12]:
MAPPED_PATH = PROCESSED_PATH / "transactions_mapped.parquet"

if not MAPPED_PATH.exists():
    transactions_mapped.sink_parquet(MAPPED_PATH, compression="zstd")

In [13]:
print(f"Mapped transactions: {MAPPED_PATH.stat().st_size / 1024**3:.2f} GB")

Mapped transactions: 0.17 GB


In [14]:
VAL_START = date(2020, 9, 9)
TEST_START = date(2020, 9, 16)

In [15]:
transactions_mapped = pl.scan_parquet(MAPPED_PATH)

train = transactions_mapped.filter(pl.col("t_dat") < VAL_START)
validation = transactions_mapped.filter(
    (pl.col("t_dat") >= VAL_START) & (pl.col("t_dat") < TEST_START)
)
test = transactions_mapped.filter(pl.col("t_dat") >= TEST_START)

In [16]:
print("Train:", train.select(pl.len()).collect().item())
print("Validation:", validation.select(pl.len()).collect().item())
print("Test:", test.select(pl.len()).collect().item())

Train: 31292772
Validation: 255241
Test: 240311


In [17]:
train.sink_parquet(PROCESSED_PATH / "train.parquet", compression="zstd")
validation.sink_parquet(PROCESSED_PATH / "validation.parquet", compression="zstd")
test.sink_parquet(PROCESSED_PATH / "test.parquet", compression="zstd")

In [18]:
validation_ground_truth = (
    validation
    .group_by("customer_idx")
    .agg(pl.col("article_idx").unique().alias("actual"))
)

test_ground_truth = (
    test
    .group_by("customer_idx")
    .agg(pl.col("article_idx").unique().alias("actual"))
)

In [19]:
validation_ground_truth.head().collect()

customer_idx,actual
u32,list[u32]
717563,"[95802, 101720, … 56698]"
329039,"[104452, 3518]"
1290571,"[97561, 103795, … 88270]"
602335,"[96835, 90079, … 99400]"
396134,"[103704, 104148, … 95783]"


In [20]:
test_ground_truth.head().collect()

customer_idx,actual
u32,list[u32]
367717,"[49962, 82691]"
1157496,"[25800, 69720]"
772070,[102349]
680185,[102123]
460950,[101719]


In [21]:
print("Validation users:", validation_ground_truth.select(pl.len()).collect().item())
print("Test users:", test_ground_truth.select(pl.len()).collect().item())

Validation users: 72019
Test users: 68984


In [22]:
validation_ground_truth.select(
    pl.col("actual").list.len().min().alias("min"),
    pl.col("actual").list.len().median().alias("median"),
    pl.col("actual").list.len().quantile(0.90).alias("q90"),
    pl.col("actual").list.len().max().alias("max")
).collect()

min,median,q90,max
u32,f64,f64,u32
1,2.0,6.0,47


In [23]:
validation_ground_truth.sink_parquet(PROCESSED_PATH / "validation_ground_truth.parquet", compression="zstd")
test_ground_truth.sink_parquet(PROCESSED_PATH / "test_ground_truth.parquet", compression="zstd")

In [24]:
for name in [
    "transactions.parquet",
    "transactions_mapped.parquet",
    "customer_mapping.parquet",
    "article_mapping.parquet",
    "train.parquet",
    "validation.parquet",
    "test.parquet",
    "validation_ground_truth.parquet",
    "test_ground_truth.parquet"
]:
    path = PROCESSED_PATH / name
    print(name, f"{path.stat().st_size / 1024**2:.2f} MB")

transactions.parquet 411.47 MB
transactions_mapped.parquet 172.53 MB
customer_mapping.parquet 48.09 MB
article_mapping.parquet 0.44 MB
train.parquet 169.71 MB
validation.parquet 1.45 MB
test.parquet 1.38 MB
validation_ground_truth.parquet 0.75 MB
test_ground_truth.parquet 0.70 MB
